In [1]:

from oqd_compiler_infrastructure import Post, PrettyPrint
from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog
from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
from oqd_analog_emulator.interpreter import QutipInterpreter

printer = Post(PrettyPrint())

source = """ 
r = qreg(5)
q0 = r[0]
q1 = r[1]
q2 = r[2]
q3 = r[3]
initialize(r)
// s = [q0, q1]
s = [q1, q0]
u = [q2, q3]
t = [q0, q2]
// result = evolve(%X , 1, q0)
result = evolve( %X %@ %X, 1, s)
result2 = evolve( %X %@ %X, 1, u)
result3 = evolve( %X %@ %X, 1, t)
// evolve(%X, 1, r[2])
"""

# source= "r = qreg(2) \n initialize(r[0])"


circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
checker = AnalogTypeChecker(cfg)

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table

circuit, cfg = compile_analog_circuit(circuit=circuit, cfg=cfg, symbol_table=symbol_table)


In [2]:
interpreter = QutipInterpreter(graph=cfg) 
interpreter.run()

INIT


[]

In [3]:
instructions = interpreter.get_instructions()
instructions

[QutipBackendInstructions(class_='QutipBackendInstructions', instructions=[QutipBackendInstruction(class_='QutipBackendInstruction', opcode=<OpCode.QREG: 25>, args=['r', 5])]),
 QutipBackendInstructions(class_='QutipBackendInstructions', instructions=[QutipBackendInstruction(class_='QutipBackendInstruction', opcode=<OpCode.GLOBAL: 0>, args=['q0']), QutipBackendInstruction(class_='QutipBackendInstruction', opcode=<OpCode.EXTRACT: 2>, args=['r', 0]), QutipBackendInstruction(class_='QutipBackendInstruction', opcode=<OpCode.STORE: 4>, args=['q0'])]),
 QutipBackendInstructions(class_='QutipBackendInstructions', instructions=[QutipBackendInstruction(class_='QutipBackendInstruction', opcode=<OpCode.GLOBAL: 0>, args=['q1']), QutipBackendInstruction(class_='QutipBackendInstruction', opcode=<OpCode.EXTRACT: 2>, args=['r', 1]), QutipBackendInstruction(class_='QutipBackendInstruction', opcode=<OpCode.STORE: 4>, args=['q1'])]),
 QutipBackendInstructions(class_='QutipBackendInstructions', instructio

In [4]:
store = interpreter.get_store()
store
# interpreter.vm.stack
# store['q0'].state

{'r': [<ListTerminators.LISTSTART: 0>,
  RegisterObject(name='r', index=0),
  RegisterObject(name='r', index=1),
  RegisterObject(name='r', index=2),
  RegisterObject(name='r', index=3),
  RegisterObject(name='r', index=4),
  <ListTerminators.LISTEND: 1>],
 'q0': RegisterObject(name='r', index=0),
 'q1': RegisterObject(name='r', index=1),
 'q2': RegisterObject(name='r', index=2),
 'q3': RegisterObject(name='r', index=3),
 's': [<ListTerminators.LISTSTART: 0>,
  RegisterObject(name='r', index=1),
  RegisterObject(name='r', index=0),
  <ListTerminators.LISTEND: 1>],
 'u': [<ListTerminators.LISTSTART: 0>,
  RegisterObject(name='r', index=2),
  RegisterObject(name='r', index=3),
  <ListTerminators.LISTEND: 1>],
 't': [<ListTerminators.LISTSTART: 0>,
  RegisterObject(name='r', index=0),
  RegisterObject(name='r', index=2),
  <ListTerminators.LISTEND: 1>],
 'result': [<ListTerminators.LISTSTART: 0>, <ListTerminators.LISTEND: 1>],
 'result2': [<ListTerminators.LISTSTART: 0>, <ListTerminators.

In [5]:
# from oqd_core.compiler.analog.math.rules import SubstituteMathVar
# from oqd_core.interface.analog.expr import MathVar
# from oqd_compiler_infrastructure import Post
# substitute_pass = Post(SubstituteMathVar(MathVar(class_='MathVar', name='#s'), MathVar(class_='MathVar', name='#t') - 10))

# substitute_pass(store['a'])
registers = interpreter.vm.registers
registers

{RegisterObject(name='r', index=0): QubitRegister(name=[RegisterObject(name='r', index=1), RegisterObject(name='r', index=0), RegisterObject(name='r', index=2), RegisterObject(name='r', index=3)], time=3.0, state=Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
 Qobj data =
 [[ 0.15772839+0.j        ]
  [ 0.        +0.j        ]
  [ 0.        +0.j        ]
  [ 0.        -0.24564757j]
  [ 0.        +0.j        ]
  [ 0.        -0.24564757j]
  [-0.38257367+0.j        ]
  [ 0.        +0.j        ]
  [ 0.        +0.j        ]
  [-0.38257367+0.j        ]
  [ 0.        -0.24564757j]
  [ 0.        +0.j        ]
  [ 0.        +0.59582357j]
  [ 0.        +0.j        ]
  [ 0.        +0.j        ]
  [-0.38257367+0.j        ]]),
 RegisterObject(name='r', index=1): QubitRegister(name=[RegisterObject(name='r', index=1), RegisterObject(name='r', index=0), RegisterObject(name='r', index=2), RegisterObject(name='r', index=3)], time=3.0, state=Quantum object: dims=[[2, 2, 

In [6]:
cfg.to_dict()

{0: {'register_id': 0,
  'kind': 'start',
  'stmt': {},
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'r',
   'value': {'class_': 'QuantumRegister', 'size': 5}},
  'preds': [0],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 2: {'register_id': 2,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q0',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 0}},
  'preds': [1],
  'succs': [3],
  'exit_nodes': [],
  'edge_labels': {}},
 3: {'register_id': 3,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q1',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 1}},
  'preds': [2],
  'succs': [4],
  'exit_nodes': [],
  'edge_labels': {}},
 4: {'register_id': 4,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q2',
   'value': {'cla